# 08 · Tabelas e figuras finais

Junta os resultados oficiais das etapas 01 a 07 em tabelas e figuras prontas para o texto do TCC.
Não treina nada: só lê `results/` (e os preços, para a figura 1). Roda no PC ou no Kaggle, sem GPU.

In [ ]:
import os, sys, glob
achados = glob.glob('../src/tcc/config.py') + glob.glob('/kaggle/input/**/src/tcc/config.py', recursive=True)
RAIZ = achados[0].rsplit('/src/', 1)[0]              # pasta do projeto (no PC ou no dataset do Kaggle)
sys.path.insert(0, os.path.join(RAIZ, 'src'))
import warnings; warnings.filterwarnings('ignore')
import json, shutil
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from tcc import config, dados, experimento

In [ ]:
INICIO, FIM = '2022-01-01', '2023-12-31'
MOEDAS = config.MOEDAS
ETAPAS = {1: '01_alvo', 2: '02_lstm', 3: '03_transformer', 4: '04_sentimento',
          5: '05_com_sentimento', 6: '06_controle', 7: '07_walk_forward'}
ler = lambda etapa, arquivo, **kw: pd.read_csv(os.path.join(RAIZ, 'results', ETAPAS[etapa], arquivo), **kw)

SAIDA = config.PASTA_RESULTADOS / '08_tabelas_figuras'
(SAIDA / 'tabelas').mkdir(parents=True, exist_ok=True)
(SAIDA / 'figuras').mkdir(parents=True, exist_ok=True)

# Estilo das figuras: cores fixas por versão, naive sempre cinza tracejado
COR = {'financeiro': '#2a78d6', 'sentimento_ffill': '#eb6834', 'sentimento_copia_btc': '#1baf7a', 'naive': '#52514e'}
NOME = {'financeiro': 'Só OHLCV', 'sentimento_ffill': 'OHLCV + sentimento (ffill)', 'sentimento_copia_btc': 'OHLCV + sentimento (BTC)'}
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 200, 'font.size': 10, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25, 'axes.axisbelow': True})
tabelas = {}

def salvar_tabela(nome, df):
    tabelas[nome] = df
    df.to_csv(SAIDA / 'tabelas' / f'{nome}.csv')
    return df

def salvar_figura(nome):
    plt.tight_layout(); plt.savefig(SAIDA / 'figuras' / f'{nome}.png', bbox_inches='tight'); plt.show()

def virgula(x, casas=2):
    return f'{x:.{casas}f}'.replace('.', ',')

## Tabela 1 · Alvo: preço ou retorno (etapa 01)

In [ ]:
r = ler(1, 'resumo.csv')
t = r.pivot(index='moeda', columns='alvo', values='MAPE_%').loc[MOEDAS, ['preco', 'retorno']]
t.insert(0, 'naive', r.groupby('moeda')['MAPE_naive_%'].first())
t['p_preco_vs_naive'] = r[r.alvo == 'preco'].set_index('moeda')['p_vs_naive']
t['p_retorno_vs_naive'] = r[r.alvo == 'retorno'].set_index('moeda')['p_vs_naive']
t.loc['Média', ['naive', 'preco', 'retorno']] = t[['naive', 'preco', 'retorno']].mean()
salvar_tabela('t1_alvo_preco_vs_retorno', t.round(3))

## Tabela 2 · Hiperparâmetros finais (etapas 02 e 03)

In [ ]:
hp = {nome: json.load(open(os.path.join(RAIZ, 'results', ETAPAS[e], 'hiperparametros.json'))) for nome, e in [('LSTM', 2), ('Transformer', 3)]}
t = pd.DataFrame(hp).drop(index='arquitetura', errors='ignore')
t.loc['melhor_trial_MAPE_rede/naive'] = [ler(e, 'trials.csv')['value'].min() for e in (2, 3)]
salvar_tabela('t2_hiperparametros', t)

## Tabela 3 · Textos, cobertura e correlação com o retorno (etapa 04)

In [ ]:
cob = ler(4, 'cobertura.csv').pivot(index='moeda', columns='fonte', values=['textos', 'cobertura_%']).loc[MOEDAS]
cor = ler(4, 'correlacao_retorno.csv').pivot_table(index='moeda', columns=['fonte', 'retorno'], values=['rho', 'p']).loc[MOEDAS]
cob.columns = [f'{a}_{b}' for a, b in cob.columns]
cor.columns = [f'{a}_{b}_{c}' for a, b, c in cor.columns]
t = pd.concat([cob, cor], axis=1)
salvar_tabela('t3_sentimento_cobertura_correlacao', t.round(3))

## Tabela 4 · Resultado principal: o sentimento melhora a previsão? (etapa 05)

In [ ]:
r = ler(5, 'resumo.csv')
dm = ler(5, 'diebold_mariano.csv')
r = r.merge(dm[['moeda', 'modelo', 'variante', 'p']].rename(columns={'p': 'p_vs_financeiro'}), how='left')
t = r.set_index(['modelo', 'variante', 'moeda'])[['MAPE_%', 'desvio_sementes', 'MAPE_naive_%', 'RMSE', 'acerto_direcao_%', 'p_vs_naive', 'p_vs_financeiro']]
t = t.loc[[(mo, v, m) for mo in ['LSTM', 'Transformer'] for v in NOME for m in MOEDAS]]
salvar_tabela('t4_resultado_principal', t.round(4))

mape = r.pivot_table(index='moeda', columns=['modelo', 'variante'], values='MAPE_%').loc[MOEDAS]
mape = mape[[(mo, v) for mo in ['LSTM', 'Transformer'] for v in NOME]]
mape.insert(0, ('naive', ''), r.groupby('moeda')['MAPE_naive_%'].first())
mape.loc['Média'] = mape.mean()
salvar_tabela('t5_mape_resumido', mape.round(3))

## Tabela 6 · Controle com sentimento sintético (etapa 06)

In [ ]:
r = ler(6, 'resumo.csv')
c6 = ler(6, 'correlacao_sinal.csv').groupby('variante')['rho_amanha'].mean()
t = r.groupby(['variante', 'modelo'])[['MAPE_%', 'acerto_direcao_%']].mean().unstack('modelo')
t.columns = [f'{a}_{b}' for a, b in t.columns]
t.insert(0, 'spearman_com_amanha', c6)
ordem = ['financeiro', 'hoje', 'amanha_1.0', 'amanha_0.5', 'amanha_0.25', 'amanha_0.1']
t = t.loc[ordem]
t.loc['naive', ['MAPE_%_LSTM', 'MAPE_%_Transformer']] = r.groupby('moeda')['MAPE_naive_%'].first().mean()
salvar_tabela('t6_controle_sintetico', t.round(3))

## Tabela 7 · Walk-forward por regime (etapa 07)

In [ ]:
r = ler(7, 'resumo.csv')
regimes = list(config.REGIMES) + ['Todos']
t = r.groupby(['regime', 'modelo', 'variante'])['razao_naive'].mean().unstack(['modelo', 'variante']).loc[regimes]
t = t[[(mo, v) for mo in ['LSTM', 'Transformer'] for v in NOME]]
t.insert(0, ('naive_MAPE_%', ''), r.groupby(['regime', 'moeda'])['MAPE_naive_%'].first().groupby('regime').mean())
dm = ler(7, 'diebold_mariano.csv')
t[('sentimento_melhor', 'p<0,05')] = dm[(dm.p < .05) & (dm.DM < 0)].groupby('regime').size().reindex(regimes, fill_value=0)
t[('sentimento_pior', 'p<0,05')] = dm[(dm.p < .05) & (dm.DM > 0)].groupby('regime').size().reindex(regimes, fill_value=0)
t[('comparacoes', '')] = dm.groupby('regime').size().reindex(regimes)
salvar_tabela('t7_walk_forward_regimes', t.round(3))

## Figura 1 · Preços e divisão do período

In [ ]:
fig, eixos = plt.subplots(len(MOEDAS), 1, figsize=(10, 9), sharex=True)
for ax, m in zip(eixos, MOEDAS):
    close = dados.precos(m, INICIO, FIM)['Close']
    i_val, i_teste = experimento.cortes(len(close) - 1)
    datas = close.index[1:]
    ax.axvspan(datas[i_val], datas[i_teste], color='#eda100', alpha=.15, lw=0)
    ax.axvspan(datas[i_teste], datas[-1], color='#e34948', alpha=.12, lw=0)
    ax.plot(close.index, close, color='#0b0b0b', lw=1.2)
    ax.set_ylabel(f'{m} (US$)')
eixos[0].text(datas[i_val], eixos[0].get_ylim()[1], ' validação', va='top', fontsize=8, color='#52514e')
eixos[0].text(datas[i_teste], eixos[0].get_ylim()[1], ' teste', va='top', fontsize=8, color='#52514e')
eixos[0].set_title('Fechamento diário, 2022–2023, com a divisão 70 / 15 / 15', loc='left')
salvar_figura('f1_precos_divisao')

## Figura 2 · Sentimento x retorno de hoje e de amanhã (etapa 04)

In [ ]:
cor = ler(4, 'correlacao_retorno.csv')
fig, eixos = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, (fonte, titulo) in zip(eixos, [('SentimentoNoticia', 'Notícias'), ('SentimentoTweet', 'Tweets')]):
    for k, (quando, cor_barra, rotulo) in enumerate([('hoje', '#2a78d6', 'retorno de hoje'), ('amanha', '#eb6834', 'retorno de amanhã')]):
        g = cor[(cor.fonte == fonte) & (cor.retorno == quando)].set_index('moeda').loc[MOEDAS]
        x = np.arange(len(MOEDAS)) + (k - .5) * .38
        ax.bar(x, g.rho, .36, color=cor_barra, label=rotulo)
        for xi, (rho, p) in zip(x, zip(g.rho, g.p)):
            if p < .05:
                ax.text(xi, rho + .005, '*', ha='center', fontsize=11)
    ax.axhline(0, color='#52514e', lw=.8)
    ax.set_xticks(range(len(MOEDAS))); ax.set_xticklabels(MOEDAS); ax.set_title(titulo, loc='left')
eixos[0].set_ylabel('Spearman (dias com texto)'); eixos[0].legend(frameon=False, fontsize=9)
fig.suptitle('O sentimento acompanha o retorno do mesmo dia, não o de amanhã  (* p < 0,05)', x=.01, ha='left', fontsize=10)
salvar_figura('f2_sentimento_vs_retorno')

## Figura 3 · Erro relativo ao naive, com e sem sentimento (etapa 05)

In [ ]:
r = ler(5, 'resumo.csv')
r['razao'] = r['MAPE_%'] / r['MAPE_naive_%']
fig, eixos = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, modelo in zip(eixos, ['LSTM', 'Transformer']):
    for k, v in enumerate(NOME):
        g = r[(r.modelo == modelo) & (r.variante == v)].set_index('moeda').loc[MOEDAS]
        ax.plot(np.arange(len(MOEDAS)) + (k - 1) * .18, g.razao, 'o', ms=8, color=COR[v], label=NOME[v],
                markeredgecolor='white', markeredgewidth=1.5)
    ax.axhline(1, color=COR['naive'], ls='--', lw=1.2, label='naive (= 1)')
    ax.set_xticks(range(len(MOEDAS))); ax.set_xticklabels(MOEDAS); ax.set_title(modelo, loc='left')
eixos[0].set_ylabel('MAPE da rede ÷ MAPE do naive')
fig.legend(*eixos[0].get_legend_handles_labels(), loc='lower center', ncol=4, frameon=False, fontsize=8, bbox_to_anchor=(.5, -.06))
salvar_figura('f3_razao_naive_com_sem_sentimento')

## Figura 4 · Quanto o sentimento precisaria antecipar (etapa 06)

In [ ]:
r6 = ler(6, 'resumo.csv'); c6 = ler(6, 'correlacao_sinal.csv').groupby('variante')['rho_amanha'].mean()
r6['razao'] = r6['MAPE_%'] / r6['MAPE_naive_%']
r5 = ler(5, 'resumo.csv'); r5['razao'] = r5['MAPE_%'] / r5['MAPE_naive_%']
cor4 = ler(4, 'correlacao_retorno.csv')
rho_real = cor4[(cor4.retorno == 'amanha') & (cor4.fonte == 'SentimentoNoticia')]['rho'].mean()
fig, ax = plt.subplots(figsize=(7, 4))
for modelo, estilo in [('LSTM', 'o-'), ('Transformer', 's-')]:
    vs = ['amanha_0.1', 'amanha_0.25', 'amanha_0.5', 'amanha_1.0']
    y = [r6[(r6.modelo == modelo) & (r6.variante == v)]['razao'].mean() for v in vs]
    ax.plot([c6[v] for v in vs], y, estilo, color='#2a78d6' if modelo == 'LSTM' else '#6250d6', ms=7, lw=2, label=f'{modelo} com sinal sintético')
    ax.plot(rho_real, r5[(r5.modelo == modelo) & (r5.variante == 'sentimento_ffill')]['razao'].mean(), estilo[0], ms=10, zorder=5,
            color='#eb6834', markeredgecolor='white', markeredgewidth=1.5, label=f'{modelo} com sentimento real (notícias, ffill)')
ax.axhline(1, color=COR['naive'], ls='--', lw=1.2, label='naive')
ax.set_xlabel('Spearman do sinal com o retorno de amanhã (teste)'); ax.set_ylabel('MAPE ÷ MAPE do naive (média das moedas)')
ax.legend(frameon=False, fontsize=8)
ax.set_title('As redes usam um sinal que antecipa o mercado; o sentimento real fica perto de 0', loc='left', fontsize=10)
salvar_figura('f4_controle_sintetico')

## Figura 5 · Walk-forward por regime (etapa 07)

In [ ]:
r = ler(7, 'resumo.csv')
regimes = list(config.REGIMES) + ['Todos']
media = r.groupby(['modelo', 'regime', 'variante'])['razao_naive'].mean()
fig, eixos = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, modelo in zip(eixos, ['LSTM', 'Transformer']):
    for k, v in enumerate(NOME):
        ax.bar(np.arange(len(regimes)) + (k - 1) * .27, [media[(modelo, rg, v)] - 1 for rg in regimes], .25, bottom=1,
               color=COR[v], label=NOME[v], edgecolor='white', linewidth=1)
    ax.axhline(1, color=COR['naive'], ls='--', lw=1.2)
    ax.set_xticks(range(len(regimes))); ax.set_xticklabels(regimes, fontsize=8); ax.set_title(modelo, loc='left')
eixos[0].set_ylabel('MAPE ÷ MAPE do naive (média das moedas)'); eixos[0].legend(frameon=False, fontsize=8)
eixos[0].set_ylim(.99, media.max() + .01)
salvar_figura('f5_walk_forward_regimes')

## Figura 6 · Exemplo de previsão: BTC no teste (etapa 05)

In [ ]:
p = ler(5, 'previsoes.csv', parse_dates=['data'])
p = p[(p.moeda == 'BTC') & (p.modelo == 'Transformer') & p.data.between('2023-10-15', '2023-11-30')]
g = p.groupby(['variante', 'data'])[['real', 'naive', 'previsto']].mean()
fig, ax = plt.subplots(figsize=(10, 3.8))
fin = g.loc['financeiro']
ax.plot(fin.index, fin.real, color='#0b0b0b', lw=2, marker='o', ms=4, label='real')
for v in ['financeiro', 'sentimento_ffill']:
    ax.plot(g.loc[v].index, g.loc[v].previsto, color=COR[v], lw=2, label=f'Transformer · {NOME[v]}')
ax.plot(fin.index, fin.naive, color=COR['naive'], ls='--', lw=1.4, zorder=5, label='naive (preço de ontem)')
ax.set_ylabel('BTC (US$)'); ax.legend(frameon=False, fontsize=8, ncol=2)
ax.set_title('BTC, 15/10 a 30/11/2023: as previsões ficam coladas no naive, com ou sem sentimento', loc='left', fontsize=10)
salvar_figura('f6_exemplo_btc_teste')

## Planilha única com todas as tabelas

In [ ]:
with pd.ExcelWriter(SAIDA / 'tabelas.xlsx') as planilha:
    for nome, df in tabelas.items():
        df.to_excel(planilha, sheet_name=nome[:31])
print('Pronto:', shutil.make_archive(str(SAIDA), 'zip', SAIDA))